### **Laboratorio 9 - Políticas de control, agente LLM y terminación verificable**

En esta sesión se inspeccionará un benchmark pequeño pero trazable. No se pretende ganar una competición de modelos: se exige justificar las decisiones de control, identificar fallos y reconocer los límites de las métricas. Este laboratorio utiliza una muestra real de SQuAD 1.1, distinta del benchmark institucional de Semana 4. Los datos, las consultas y los qrels permanecen congelados entre las condiciones A0, A1 y B.

**Consigna:** en las celdas de análisis escriba conclusiones basadas en ejecuciones propias, no afirmaciones anticipadas.

#### **Laboratorio, etapa 1. Verificación de entradas y contrato experimental**

Un benchmark que cambia entre condiciones no permite atribuir diferencias al controlador. Verificamos tamaño, unicidad de consultas, qrels y hashes. El umbral 0.45 se establece **antes** de observar resultados.

In [ ]:
from pathlib import Path
import json, os, sys
root = Path.cwd().resolve()
for directory in (root, *root.parents):
    if (directory / "Semana9/agent_benchmark.py").is_file():
        root = directory
        break
else:
    raise FileNotFoundError("Ejecuta desde la raiz de CC-0F4 con Semana9 disponible")
sys.path.insert(0, str(root))
from Semana9.agent_benchmark import (BM25Tool, jsonl, lexical_coverage,
                                     run_workflow, run_adaptive, evaluate_run)
from Semana9.agent_loop import (Limits, RestrictedTools, ScriptedPolicy, run_agent)
from Semana9.agent_demo import demo
print("Repositorio:", root)
from Semana9.agent_benchmark import real_paths
real_paths(root)  # Falla con instrucciones si no se ha preparado SQuAD real.


In [ ]:
corpus = jsonl(root / "Semana9/datos_reales/corpus.jsonl")
queries = jsonl(root / "Semana9/datos_reales/queries.jsonl")
qrels = json.loads((root / "Semana9/datos_reales/qrels.json").read_text(encoding="utf-8"))
assert len(corpus) == 8 and len(queries) == 16
assert len({p["passage_id"] for d in corpus for p in d["passages"]}) == 32
assert set(qrels) == {q["query_id"] for q in queries}
summary = evaluate_run(root, root / "Semana9/resultados", threshold=0.45)
print(json.dumps(summary["data_sha256"], ensure_ascii=False, indent=2))

#### **Laboratorio, etapa 2. Tres políticas, una misma herramienta**

A0 obtiene top-1, A1 obtiene top-3 y B amplía de top-1 a top-3 bajo una política predeclarada. Registra por consulta `evidence_hit`, `tool_calls`, `context_token_proxy`, `stop_reason` y latencia. No equipares un proxy léxico con tokens de LLM.

#### **Fundamento matemático A: cobertura léxica y bifurcación**

La función de control toma una decisión **sin consultar los qrels**. Si $T(x)$ es el conjunto de términos normalizados de una cadena:

$$
c(q,d)=\frac{|T(q)\cap T(d)|}{|T(q)|}, \qquad
k(q)=\begin{cases}1 & c(q,d_1)\geq 0.45\\3 & c(q,d_1)<0.45\end{cases}
$$

Donde $q$ es la consulta, $d_1$ el primer pasaje recuperado y $k$ la cantidad de pasajes al terminar. 

**Ejemplo aritmético de clase, no resultado de SQuAD:** si una consulta tiene 5 términos distintos y solo 2 aparecen en el primer pasaje, $c=2/5=0.40$, por lo que B amplía a `top_k=3`. Comprueba el caso real de una consulta de SQuAD en la celda siguiente.

In [ ]:
from Semana9.agent_benchmark import tokens
q0 = queries[0]["query"]
first_passage = BM25Tool(corpus).search(q0,1)[0]
intersection = set(tokens(q0)).intersection(tokens(first_passage["text"]))
coverage = lexical_coverage(q0,first_passage["text"])
print("ID:", queries[0]["query_id"], "términos en consulta:", len(set(tokens(q0))))
print("intersección:",sorted(intersection), "cobertura:",round(coverage,4))
print("decisión esperada:", "ampliar a top-3" if coverage < .45 else "mantener top-1")
assert 0.0 <= coverage <= 1.0

In [ ]:
import csv
with (root / "Semana9/resultados/runs.csv").open(encoding="utf-8", newline="") as fh:
    records = list(csv.DictReader(fh))
assert len(records) == 48
for label, metrics in summary["results"].items():
    print(label, "Hit:", metrics["evidence_hit_rate"],
          "Calls:", metrics["mean_tool_calls"],
          "Context proxy:", metrics["mean_context_token_proxy"])
print("Filas:", len(records))

#### **Análisis pareado, no únicamente medias**

Una media oculta decisiones equivocadas. Localiza las consultas donde B utiliza una segunda búsqueda y compara la cobertura con A1. El conjunto de relevantes se usa solo después de ejecutar los sistemas.

In [ ]:
by_query = {}
for r in records:
    by_query.setdefault(r["query_id"], {})[r["policy"]] = r
expanded = []
saved = []
for qid, d in by_query.items():
    b = d["adaptive_rule_based"]
    a1 = d["workflow_fixed_top3"]
    if int(b["tool_calls"]) == 2:
        expanded.append(qid)
    else:
        saved.append(qid)
    assert int(b["evidence_hit"]) <= int(a1["evidence_hit"])
print("B amplio:", len(expanded), expanded)
print("B no amplio:", len(saved), saved)
print("Nota: A1 tiene una llamada incluso cuando B necesita dos")

**Pregunta de ingeniería:** ¿si B recupera los mismos tres pasajes que A1, por qué su latencia puede ser mayor? ¿Qué variable debería medirse para justificar el ahorro de contexto? Redacte su respuesta sin atribuir causalidad a diferencias de hardware.

#### **Ejercicio obligatorio 1: explica una decisión con números**

Identifica dos consultas del benchmark: una en que B amplíe y otra en que no lo haga, si ambas existen. Para cada una presenta **ID, términos distintos de la consulta, intersección con el primer pasaje, cobertura calculada, comparación con 0.45 y decisión**. Adjunta la traza. Si no existe uno de esos grupos en la muestra, indica explícitamente sin fabricar ejemplos.

#### **Laboratorio, etapa 3. Leer cuatro trazas completas**

Inspecciona al menos una traza de búsqueda ampliada, una que termine tras una sola búsqueda, una con evidencia no pertinente y una que represente un compromiso de presupuesto. Si no existe alguno de esos casos, indica expresamente su ausencia en el conjunto real.

In [ ]:
traces = jsonl(root / "Semana9/resultados/traces.jsonl")
selected = []
for predicate in (
    lambda t: t["policy"] == "adaptive_rule_based" and t["tool_calls"] == 2,
    lambda t: t["policy"] == "adaptive_rule_based" and t["tool_calls"] == 1,
    lambda t: t["policy"] == "workflow_fixed_top3",
    lambda t: t["policy"] == "workflow_fixed_top1",
):
    found = next((t for t in traces if predicate(t)), None)
    if found:
        selected.append(found)
for t in selected:
    print("\n", t["query_id"], t["policy"], "parada:", t["stop_reason"])
    print(json.dumps(t["trace"], ensure_ascii=False, indent=2)[:1600])

#### **Laboratorio, etapa 4. Bucle de agente y límites**

Los tres experimentos previos no contienen LLM. Ahora trabajaremos primero con una **política guionizada** para probar la infraestructura. La política guionizada no es un agente dirigido por LLM: únicamente prueba los controles del bucle. Después puede activarse la política real con el modelo Qwen local, usando `CC0F4_RUN_REAL_LLM=1`, la ejecución de modelo requiere recursos y puede producir errores de formato. El reporte tiene que distinguir ambas condiciones.

#### **Fundamento matemático B: estado, política y parada**

El controlador de una ejecución mantiene un estado transitorio:

$$s_t=(t, n_{tools}, n_{invalid}, O_t, C_t, b_t),$$

con paso $t$, número de herramientas, acciones inválidas, observaciones, citas observadas y presupuesto consumido. La política propone una acción $a_t=\pi(s_t,q)$, que el controlador **valida antes de ejecutarla**. Solo una acción válida y autorizada puede producir una observación que actualice el estado:

$$s_{t+1}=F(s_t,a_t,o_{t+1}).$$

El sistema finaliza por éxito operacional **o** por presupuesto, error o repetición. No confunda `completed_with_citations` con exactitud semántica. Además, `deadline_seconds` es cooperativo, no un timeout duro.

In [ ]:
tool = BM25Tool(corpus)
q = queries[0]
first_id = tool.search(q["query"], 3)[0]["passage_id"]
restricted = RestrictedTools(tool)
script = ScriptedPolicy([
    {"action": "search_documents", "arguments": {"query": q["query"], "top_k": 3}},
    {"action": "read_passage", "arguments": {"passage_id": first_id}},
    {"action": "finish", "arguments": {"answer": "Examinar el pasaje recuperado de SQuAD antes de aceptar una respuesta.",
                                      "citations": [first_id]}},
])
demonstration = run_agent(q["query"], script, restricted,
                          Limits(max_steps=6, max_tool_calls=3))
print("Replay sin LLM:", demonstration["stop_reason"])
print("Citas verificadas:", demonstration["citations"])
assert demonstration["stop_reason"] == "completed_with_citations"

#### **Laboratorio, etapa 5. Fallos inyectados y replanning**

No es suficiente que el ejemplo funcione cuando todo va bien. El controlador debe rechazar una lectura no autorizada, impedir una respuesta con citas inventadas y detectar intentos de repetir la misma herramienta. Las acciones inválidas no deben producir efectos externos.

#### **Ejercicio obligatorio 2: justificación de la parada**

Para **cada caso de fallo**: indica la primera acción rechazada, la validación responsable (allowlist, argumento, cita o repetición), el contador afectado y la razón final de parada. Localice la condición correspondiente en `Semana9/agent_loop.py`. Escribe también por qué una instrucción en el prompt no reemplaza esa comprobación de código.

In [ ]:
cases = {
    "prohibida": [
        {"action": "run_shell", "arguments": {"command": "echo no ejecutar"}},
        {"action": "run_shell", "arguments": {"command": "echo no ejecutar"}},
    ],
    "repetida": [
        {"action": "search_documents", "arguments": {"query": q["query"], "top_k": 1}},
        {"action": "search_documents", "arguments": {"query": q["query"], "top_k": 1}},
    ],
    "cita_inventada": [
        {"action": "finish", "arguments": {"answer": "No verificado", "citations": ["NO-EXISTE"]}},
        {"action": "finish", "arguments": {"answer": "No verificado", "citations": ["NO-EXISTE"]}},
    ],
}
for name, actions in cases.items():
    result = run_agent(q["query"], ScriptedPolicy(actions), restricted)
    print(name, "=>", result["stop_reason"], "calls=", result["tool_calls"])
    assert result["stop_reason"] != "completed_with_citations"

#### **Laboratorio, etapa 6. Modelo real y observaciones**

La siguiente celda **no carga un modelo por defecto**. En modalidad real, usar modelo y versión fijados y registrar hardware, configuración, tokens observados y límites. `completed_with_citations` no significa que la respuesta sea correcta: debe revisarse con las anotaciones de Semana 7. Si el modelo no se ejecuta, no inventes resultados ni llame "ReAct real" a la simulación.

#### **Ejercicio obligatorio 3: replanning observado**

Diseña una secuencia pequeña de `ScriptedPolicy` que primero intente leer un pasaje no observado, reciba el error y luego busque, lea un pasaje permitido y finalice con una cita válida. Documenta paso por paso las observaciones, la corrección y el límite de acciones inválidas. El replay sirve para verificar la infraestructura, **no** demuestra que un LLM aprendió a replanificar.

In [ ]:
if os.getenv("CC0F4_RUN_REAL_LLM", "0") == "1":
    result_llm = demo(root, policy_name="llm", query_id="Q01", model_id=os.getenv(
        "CC0F4_AGENT_MODEL", "Qwen/Qwen2.5-0.5B-Instruct"))
    print("Modelo:", result_llm["model_id"])
    print("Terminacion:", result_llm["stop_reason"], "pasos:", result_llm["step"])
    print("Tokens de entrada/salida:", result_llm["prompt_tokens"], result_llm["generated_tokens"])
    print(json.dumps(result_llm["trace"], ensure_ascii=False, indent=2)[:5000])
else:
    print("LLM no ejecutado. No se reportan metricas, calidad ni costo de agente LLM.")

#### **Laboratorio, etapa 7. Comparación y análisis de amenazas a la validez**

Compara solamente aquello que es comparable. A0/A1/B usan BM25 fijo y difieren en control. C, si se ejecutó, permite acciones y reformulaciones adicionales, no comparte necesariamente el mismo espacio de decisiones. La salida debe separar resultados verificados de hipótesis y limitaciones.

Prepara una tabla con tasa de acierto de evidencia, llamadas, pasos, proxy de contexto y una explicación de cada cambio. No combines latencia de ejecución offline con latencia de inferencia real como si fueran la misma variable.

#### **Fundamento matemático C: comparación pareada y techo de recuperación**

Para la consulta $q$, sea $R_q$ el conjunto anotado de pasajes pertinentes y $C_q^X$ los citados por la política $X$:

$$h_X(q)=\mathbb{1}[R_q\cap C_q^X\ne\varnothing].$$

Como las listas se obtienen de **un ranking BM25 fijo** y B devuelve exactamente top-1 o top-3, se verifica para cada consulta:

$$h_{A0}(q)\le h_B(q)\le h_{A1}(q).$$

Esto no mide precisión de la respuesta. Tampoco demuestra que B sea más inteligente que A1: B puede ahorrar contexto, pero si amplía realiza una llamada adicional. Compara las 16 consultas **pareadas** y no solo las tres medias.

In [ ]:
metrics = summary["results"]
a1 = metrics["workflow_fixed_top3"]
b = metrics["adaptive_rule_based"]
print("Delta hit B-A1:", round(b["evidence_hit_rate"]-a1["evidence_hit_rate"], 6))
print("Delta llamadas B-A1:", round(b["mean_tool_calls"]-a1["mean_tool_calls"], 6))
print("Delta proxy de contexto B-A1:",
      round(b["mean_context_token_proxy"]-a1["mean_context_token_proxy"], 6))
print("La magnitud y signo de cada delta deben interpretarse usando trazas")

#### **Entrega** 
Entrega el notebook ejecutable, tres archivos del benchmark, hasta cuatro trazas seleccionadas (si existen esas categorías en la muestra), las pruebas que ejecutó y un análisis de **hasta dos páginas**. Explica además una decisión de diseño que cambió al observar una falla. Conserva los hashes del corpus y no edites las 16 preguntas originales después de ver sus resultados.

**Defensa:** explica, señalando código, por qué (a) una política adaptativa no es necesariamente un agente, (b) el controlador puede rechazar al modelo, (c) `completed_with_citations` no prueba exactitud semántica y (d) el estado transitorio usado no constituye memoria episódica.

#### **Guía de análisis para la defensa individual**

Responde con evidencia propia, indicando IDs y líneas de código: (1) ¿En qué consulta B tomó una decisión distinta de A0 y A1? (2) ¿Qué limitación lógica hace imposible que B supere a A1 en `evidence_hit` sobre el mismo ranking? (3) ¿Cómo se registra la primera acción rechazada en cada uno de los tres fallos inyectados? (4) ¿Qué observación de SQuAD y qué resultado permitirían afirmar solo *recuperación de evidencia* y no *respuesta correcta*? (5) ¿Cómo adaptarías el controlador para permitir una memoria persistente en la semana 10, sin confundirla con `RunState`?.